# 18.5 Weak supervision and programmatic labeling

Weak supervision scales labeling by combining noisy rules whose coverage and conflicts are explicit.

Labeling functions abstain, conflict, and carry different reliabilities. This notebook builds the label matrix, aggregates weighted weak labels, and trains one downstream classifier.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(18)
random.seed(18)


def dataquality_ladder():
    """D1..D5 over real Breast Cancer with progressively worse data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def split_scale(X, y, seed=0):
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test


def fit_logistic(x_train, y_train, sample_weight=None):
    model = LogisticRegression(max_iter=2000, solver="lbfgs")
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"{i}. {name}: X={X.shape}, class_counts={counts}")
    print("sample rows", np.round(rungs[0][1][:3, :4], 3).tolist())


def plot_rungs_and_metric(rungs, rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    metrics = [row["metric"] for row in rows]
    for ax, (name, X, y), metric in zip(axes[0], rungs, metrics):
        ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="coolwarm", alpha=0.65)
        ax.set_title(f"{name}\n{metric:.3f}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[1][0].plot(range(1, 6), metrics, marker="o")
    axes[1][0].set_title(title)
    axes[1][0].set_xlabel("data-quality rung")
    axes[1][0].set_ylabel(ylabel)
    axes[1][0].set_ylim(0.0, 1.05)
    for ax in axes[1][1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def make_lf_matrix(X):
    radius = X[:, 0]
    texture = X[:, 1]
    perimeter = X[:, 2]
    lf1 = np.full(X.shape[0], -1, dtype=int)
    lf2 = np.full(X.shape[0], -1, dtype=int)
    lf3 = np.full(X.shape[0], -1, dtype=int)
    lf1[radius < np.quantile(radius, 0.45)] = 1
    lf1[radius > np.quantile(radius, 0.75)] = 0
    lf2[texture < np.quantile(texture, 0.45)] = 1
    lf2[texture > np.quantile(texture, 0.75)] = 0
    lf3[perimeter < np.quantile(perimeter, 0.55)] = 1
    lf3[perimeter >= np.quantile(perimeter, 0.55)] = 0
    return np.column_stack([lf1, lf2, lf3])


## The concept, built once (D1)

Let $L\in\{-1,0,1\}^{m	imes K}$, where $-1$ is abstain. Coverage is $c_k=rac{1}{m}\sum_i\mathbf{1}[L_{ik}
e -1]$, and conflict means a row has non-abstaining 0 and 1 labels. Weighted aggregation preserves reliability and abstention math.

In [ ]:
def lf_stats(L):
    coverage = np.mean(L != -1, axis=0)
    conflicts = []
    for row in L:
        spoken = row[row != -1]
        conflicts.append(len(set(spoken.tolist())) > 1)
    return coverage, float(np.mean(conflicts))


def weighted_aggregate(L, weights):
    labels = []
    confidence = []
    for row in L:
        pos = 0.0
        total = 0.0
        for value, weight in zip(row, weights):
            if value == -1:
                continue
            total += weight
            if value == 1:
                pos += weight
        score = 0.5 if total == 0 else pos / total
        labels.append(int(score >= 0.5))
        confidence.append(abs(score - 0.5) * 2)
    return np.array(labels), np.array(confidence)


def weak_label_train(X, y, seed=0, grouped=False):
    L = make_lf_matrix(X)
    weights = np.array([0.9, 0.8, 0.6])
    if grouped:
        weights = np.array([0.45, 0.4, 0.6])
    labels, confidence = weighted_aggregate(L, weights)
    sample_weight = 0.25 + confidence
    x_train, x_test, y_train, y_test = split_scale(X, y, seed=seed)
    label_train, _ = train_test_split(labels, test_size=0.35, random_state=seed, stratify=y)
    weight_train, _ = train_test_split(sample_weight, test_size=0.35, random_state=seed, stratify=y)
    model = fit_logistic(x_train, label_train, sample_weight=weight_train)
    preds = model.predict(x_test)
    label_quality = accuracy_score(y, labels, sample_weight=sample_weight)
    return accuracy_score(y_test, preds) * label_quality

Assert coverage, conflict, majority labels, and weighted confidence from the lesson.

In [ ]:
L_lesson = np.array([
    [1, 1, 1],
    [1, -1, 1],
    [0, 0, 0],
    [-1, 0, 0],
    [1, 0, 1],
])
coverage, conflict = lf_stats(L_lesson)
majority_labels = []
for row in L_lesson:
    spoken = row[row != -1]
    majority_labels.append(int(spoken.mean() >= 0.5))
weights = np.array([0.9, 0.8, 0.4])
labels, confidence = weighted_aggregate(L_lesson, weights)
weighted_score = (0.9 + 0.8) / (0.9 + 0.8 + 0.4)
print(np.round(coverage, 3).tolist(), round(conflict, 3), majority_labels)
print(round(weighted_score, 3), labels.tolist())
assert np.round(coverage, 3).tolist() == [0.8, 0.8, 1.0]
assert round(conflict, 3) == 0.2
assert majority_labels == [1, 1, 0, 0, 1]
assert round(weighted_score, 3) == 0.81

## The dataset ladder

Every notebook uses the same real Breast Cancer base table and changes data quality only: clean, label-noisy, imbalanced, feature-noisy, then missing-and-imputed. The model stays fixed so movement in the curve is caused by the data.

In [ ]:
rungs = dataquality_ladder()
preview_ladder(rungs)

In [ ]:
rows = []
for name, X, y in rungs:
    metric = weak_label_train(X, y, seed=6, grouped=False)
    rows.append({"name": name, "metric": metric})
    L = make_lf_matrix(X)
    coverage, conflict = lf_stats(L)
    print(f"{name:28s} weak_label_metric={metric:.3f} coverage={coverage.mean():.3f} conflict={conflict:.3f}")

In [ ]:
plot_rungs_and_metric(rungs, rows, "Weak-label downstream metric")

## Pitfall on D5: hidden correlated errors

Two keyword-like rules built from related signals are not independent annotators. The fix groups or down-weights correlated labeling functions and preserves abstentions instead of treating them as class 0.

In [ ]:
name, X, y = rungs[-1]
overconfident = weak_label_train(X, y, seed=8, grouped=False)
grouped = weak_label_train(X, y, seed=8, grouped=True)
L = make_lf_matrix(X)
correlated_agreement = np.mean(L[:, 0] == L[:, 2])
wrong_abstain_as_zero = L.copy()
wrong_abstain_as_zero[wrong_abstain_as_zero == -1] = 0
wrong_labels, _ = weighted_aggregate(wrong_abstain_as_zero, np.array([0.9, 0.8, 0.6]))
right_labels, _ = weighted_aggregate(L, np.array([0.45, 0.4, 0.6]))
print(f"lf1-lf3 raw agreement={correlated_agreement:.3f}")
print(f"overconfident metric={overconfident:.3f} grouped metric={grouped:.3f}")
print(f"labels changed by preserving abstain={np.mean(wrong_labels != right_labels):.3f}")
assert np.mean(wrong_labels != right_labels) > 0.0
assert grouped >= overconfident - 0.12

## Evaluate it + Practice

- Metric: weighted weak-label quality times downstream accuracy; compare with the no-skill majority-class baseline.
- Cheap sanity check: coverage counts should be 4/5, 4/5, and 5/5 on the lesson matrix.
- Ablation: treat abstain as class 0 and compare the label distribution.
- Failure signals: high conflict, high correlated-rule agreement, or confidence unsupported by coverage.

Practice prompts:
1. Change one corruption level in `dataquality_ladder()` and predict the metric direction.


In [ ]:
# Your turn: change one corruption and rerun the ladder table.


2. Add one slice check that could catch a global-average pitfall.

In [ ]:
# Your turn: define a slice and compare its metric with the global metric.


3. Replace the fixed logistic model with another CPU-safe classifier and explain whether the data-quality ordering changed.

In [ ]:
# Your turn: try a second model without changing the data-cleaning method.
